In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from math import radians

np.random.seed(42)

In [2]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="jnj_commute_analysis")
address = "Robert-Koch-Straße 1, 22851 Norderstedt, Germany"
location = geolocator.geocode(address)

WORKPLACE = {
    "name": "J&J Medical GmbH, Norderstedt",
    "lat": location.latitude,
    "lon": location.longitude,
}
workplace_point = Point(WORKPLACE["lon"], WORKPLACE["lat"])
print(WORKPLACE)
print(workplace_point)

{'name': 'J&J Medical GmbH, Norderstedt', 'lat': 53.6865222, 'lon': 10.0469639}
POINT (10.0469639 53.6865222)


In [3]:
N_EMPLOYEES = 300
HAMBURG_CENTER = {"lat": 53.5511, "lon": 9.9937}

def generate_synthetic_employees(n, center_lat, center_lon, max_radius_km=25):
    radii = max_radius_km * np.sqrt(np.random.uniform(0, 1, n))
    angles = np.random.uniform(0, 2 * np.pi, n)
    dlat = (radii * np.cos(angles)) / 111.0
    dlon = (radii * np.sin(angles)) / (111.0 * np.cos(radians(center_lat)))
    lats = center_lat + dlat
    lons = center_lon + dlon
    df = pd.DataFrame({
        "employee_id": [f"EMP{i:04d}" for i in range(1, n + 1)],
        "home_lat": lats,
        "home_lon": lons,
    })
    geometry = [Point(xy) for xy in zip(df.home_lon, df.home_lat)]
    return gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

employees_gdf = generate_synthetic_employees(N_EMPLOYEES, HAMBURG_CENTER["lat"], HAMBURG_CENTER["lon"])
employees_gdf.head()

,employee_id,home_lat,home_lon,geometry
0,EMP0001,53.681734,10.067722,POINT (10.06772 53.68173)
1,EMP0002,53.335743,9.921349,POINT (9.92135 53.33574)
2,EMP0003,53.364651,9.911786,POINT (9.91179 53.36465)
3,EMP0004,53.437867,9.770741,POINT (9.77074 53.43787)
4,EMP0005,53.537786,9.845645,POINT (9.84565 53.53779)


In [4]:
import requests

overpass_url = "https://overpass-api.de/api/interpreter"
query = f"""
[out:json][timeout:60];
(
  node["railway"="station"](around:30000,{WORKPLACE['lat']},{WORKPLACE['lon']});
  node["railway"="halt"](around:30000,{WORKPLACE['lat']},{WORKPLACE['lon']});
  node["station"="subway"](around:30000,{WORKPLACE['lat']},{WORKPLACE['lon']});
);
out body;
"""
response = requests.get(overpass_url, params={"data": query})
data = response.json()
print(f"Stations found: {len(data['elements'])}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

**Note:** The Overpass API (both the primary endpoint and a mirror) 
returned timeout/server errors during testing. Rather than depend on 
an unreliable third-party service, this analysis falls back to a 
curated real-station dataset covering the Hamburg metro area, 
compiled from public HVV/OpenStreetMap transit maps. In a production 
setting, this call would be wrapped in retry logic with the curated 
list as an automatic fallback.

In [8]:
# Curated set of real HVV stations across the Hamburg metro area
# (used as a fallback since the free Overpass API was unreliable).
# Coordinates sourced from public HVV/OpenStreetMap transit maps.
hvv_stations = [
    ("Norderstedt Mitte", 53.6975, 9.9958, "U1/AKN"),
    ("Garstedt", 53.6870, 9.9976, "U1"),
    ("Richtweg", 53.6768, 10.0027, "U1"),
    ("Kiwittsmoor", 53.6667, 10.0013, "U1"),
    ("Ochsenzoll", 53.6559, 10.0058, "U1"),
    ("Langenhorn Nord", 53.6461, 10.0089, "U1"),
    ("Langenhorn Markt", 53.6355, 10.0134, "U1"),
    ("Fuhlsbuttel Nord", 53.6262, 10.0122, "U1"),
    ("Ohlsdorf", 53.6111, 10.0221, "U1/S1/S11"),
    ("Sengelmannstrasse", 53.6032, 10.0294, "U1/U5"),
    ("Alsterdorf", 53.5966, 10.0176, "U1"),
    ("Hudtwalckerstrasse", 53.5942, 9.9977, "U1"),
    ("Kellinghusenstrasse", 53.5859, 9.9847, "U1/U3"),
    ("Klosterstern", 53.5762, 9.9868, "U1"),
    ("Friedrichsgabe", 53.7057, 9.9581, "AKN A1"),
    ("Tanneneck", 53.7157, 9.9459, "AKN A1"),
    ("Hamburg Hauptbahnhof", 53.5528, 10.0069, "S/U/Rail hub"),
    ("Altona", 53.5525, 9.9352, "S/Rail hub"),
    ("Dammtor", 53.5605, 9.9906, "S1/S21/S31"),
    ("Sternschanze", 53.5631, 9.9639, "S11/S21/S31/U3"),
    ("Landungsbrucken", 53.5463, 9.9682, "S1/S3/U3"),
    ("Berliner Tor", 53.5535, 10.0261, "S1/U1/U2/U3"),
    ("Wandsbek Markt", 53.5763, 10.0783, "U1"),
    ("Barmbek", 53.5877, 10.0396, "U2/U3/S-Bahn"),
    ("Jungfernstieg", 53.5533, 9.9937, "U1/U2/U4/S-Bahn"),
    ("Harburg", 53.4614, 9.9847, "S3/S31/Rail"),
    ("Bergedorf", 53.4886, 10.2126, "S21/Rail"),
    ("Wilhelmsburg", 53.4967, 10.0090, "S3/S31"),
    ("Blankenese", 53.5647, 9.7997, "S1"),
    ("Pinneberg", 53.6613, 9.7935, "S3/AKN"),
    ("Rahlstedt", 53.6034, 10.1466, "S4/Rail"),
    ("Poppenbuttel", 53.6438, 10.0777, "U1"),
    ("Volksdorf", 53.6403, 10.1332, "U1"),
    ("Farmsen", 53.6122, 10.1096, "U1"),
    ("Meiendorfer Weg", 53.6280, 10.1191, "U1"),
    ("Niendorf Nord", 53.6390, 9.9540, "U2"),
    ("Schnelsen", 53.6323, 9.9296, "AKN A2"),
    ("Eidelstedt", 53.6031, 9.9202, "S21/AKN"),
    ("Stellingen", 53.5866, 9.9219, "S21"),
]

stations_df = pd.DataFrame(hvv_stations, columns=["station_name", "lat", "lon", "lines"])
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=[Point(xy) for xy in zip(stations_df.lon, stations_df.lat)],
    crs="EPSG:4326",
)
print(f"{len(stations_gdf)} stations loaded")
stations_gdf.head()

39 stations loaded


,station_name,lat,lon,lines,geometry
0,Norderstedt Mitte,53.6975,9.9958,U1/AKN,POINT (9.9958 53.6975)
1,Garstedt,53.6870,9.9976,U1,POINT (9.9976 53.687)
2,Richtweg,53.6768,10.0027,U1,POINT (10.0027 53.6768)
3,Kiwittsmoor,53.6667,10.0013,U1,POINT (10.0013 53.6667)
4,Ochsenzoll,53.6559,10.0058,U1,POINT (10.0058 53.6559)


In [9]:
employees_m = employees_gdf.to_crs(epsg=25832)
stations_m = stations_gdf.to_crs(epsg=25832)
workplace_m = gpd.GeoSeries([workplace_point], crs="EPSG:4326").to_crs(epsg=25832).iloc[0]
print(employees_m.crs)

EPSG:25832


In [10]:
nearest = gpd.sjoin_nearest(employees_m, stations_m, distance_col="dist_to_home_station_m")
nearest = nearest.drop_duplicates(subset="employee_id")
nearest = nearest.rename(columns={"station_name": "nearest_home_station"})
nearest[["employee_id", "nearest_home_station", "dist_to_home_station_m"]].head()

,employee_id,nearest_home_station,dist_to_home_station_m
0,EMP0001,Poppenbuttel,4271.719865
1,EMP0002,Harburg,14600.813726
2,EMP0003,Harburg,11804.620271
3,EMP0004,Blankenese,14240.992096
4,EMP0005,Blankenese,4269.958589


In [11]:
work_point_gdf = gpd.GeoDataFrame({"id": [0]}, geometry=[workplace_m], crs=stations_m.crs)
work_nearest = gpd.sjoin_nearest(work_point_gdf, stations_m, distance_col="dist_workplace_to_station_m")
workplace_station_name = work_nearest.iloc[0]["station_name"]
workplace_station_dist_m = work_nearest.iloc[0]["dist_workplace_to_station_m"]
print(workplace_station_name, f"{workplace_station_dist_m:.0f} m")

Richtweg 3117 m


In [12]:
stations_m_indexed = stations_m.set_index("station_name")
DETOUR_FACTOR = 1.3  # real transit lines aren't straight lines

def transit_distance_km(row):
    home_station = stations_m_indexed.loc[row["nearest_home_station"], "geometry"]
    work_station = stations_m_indexed.loc[workplace_station_name, "geometry"]
    d_m = home_station.distance(work_station)
    return (d_m * DETOUR_FACTOR) / 1000

nearest["transit_distance_km"] = nearest.apply(transit_distance_km, axis=1)
nearest[["employee_id", "nearest_home_station", "transit_distance_km"]].head()

,employee_id,nearest_home_station,transit_distance_km
0,EMP0001,Poppenbuttel,8.018687
1,EMP0002,Harburg,31.193084
2,EMP0003,Harburg,31.193084
3,EMP0004,Blankenese,23.824334
4,EMP0005,Blankenese,23.824334


In [13]:
WALK_SPEED_KMH = 4.5
FIXED_WAIT_TRANSFER_MIN = 8   # avg headway/2 + interchange allowance
LAST_MILE_BUS_SPEED_KMH = 18  # workplace is 3.1km from station — too far to walk, so treat as a connecting bus

def tiered_transit_speed_kmh(distance_km):
    # Real HVV trips get faster per km over longer distances: short hops
    # are U-Bahn/bus with frequent stops (~25 km/h), longer trips
    # increasingly use S-Bahn/regional segments with higher line speed.
    if distance_km <= 10:
        return 25
    elif distance_km <= 20:
        return 32
    else:
        return 40

nearest["walk_to_station_min"] = (nearest["dist_to_home_station_m"] / 1000) / WALK_SPEED_KMH * 60
nearest["transit_speed_kmh"] = nearest["transit_distance_km"].apply(tiered_transit_speed_kmh)
nearest["transit_time_min"] = nearest["transit_distance_km"] / nearest["transit_speed_kmh"] * 60
nearest["last_mile_to_office_min"] = (workplace_station_dist_m / 1000) / LAST_MILE_BUS_SPEED_KMH * 60

nearest["total_commute_min"] = (
    nearest["walk_to_station_min"]
    + nearest["transit_time_min"]
    + FIXED_WAIT_TRANSFER_MIN
    + nearest["last_mile_to_office_min"]
)

nearest[["employee_id", "nearest_home_station", "walk_to_station_min",
         "transit_time_min", "last_mile_to_office_min", "total_commute_min"]].round(1).head(8)

,employee_id,nearest_home_station,walk_to_station_min,transit_time_min,last_mile_to_office_min,total_commute_min
0,EMP0001,Poppenbuttel,57.0,19.2,10.4,94.6
1,EMP0002,Harburg,194.7,46.8,10.4,259.9
2,EMP0003,Harburg,157.4,46.8,10.4,222.6
3,EMP0004,Blankenese,189.9,35.7,10.4,244.0
4,EMP0005,Blankenese,56.9,35.7,10.4,111.1
5,EMP0006,Niendorf Nord,15.0,16.5,10.4,50.0
6,EMP0007,Wilhelmsburg,21.8,39.1,10.4,79.3
7,EMP0008,Bergedorf,97.3,49.0,10.4,164.7


### Refining the home-to-station access assumption

The initial walk-only assumption produced unrealistic results — some 
employees were over 14km from their nearest station, implying a 
3+ hour walk. In reality, people don't walk that far to reach transit; 
they cycle or drive to a park & ride station instead.

To fix this, access time is now modeled based on distance:
- **≤ 1.5 km:** walk (4.5 km/h)
- **1.5–5 km:** bike (15 km/h)
- **> 5 km:** drive to station / park & ride (35 km/h)

These thresholds reflect typical walk/bike catchment radii used in 
public transport accessibility studies. This also feeds into the 
Deutschlandticket adoption scoring later: someone who has to drive to 
a station is a weaker candidate for ticket adoption than someone who 
can walk, even if total commute time is similar.

In [14]:
def home_access_time_min(dist_m):
    dist_km = dist_m / 1000
    if dist_km <= 1.5:
        return dist_km / 4.5 * 60, "walk"        # short distance: walk
    elif dist_km <= 5:
        return dist_km / 15 * 60, "bike"          # medium: bike
    else:
        return dist_km / 35 * 60, "drive (park & ride)"  # far: drive to station

access_results = nearest["dist_to_home_station_m"].apply(home_access_time_min)
nearest["walk_to_station_min"] = access_results.apply(lambda x: x[0])
nearest["home_access_mode"] = access_results.apply(lambda x: x[1])

nearest["total_commute_min"] = (
    nearest["walk_to_station_min"]
    + nearest["transit_time_min"]
    + FIXED_WAIT_TRANSFER_MIN
    + nearest["last_mile_to_office_min"]
)

nearest[["employee_id", "nearest_home_station", "dist_to_home_station_m",
         "home_access_mode", "walk_to_station_min", "total_commute_min"]].round(1).head(8)

,employee_id,nearest_home_station,dist_to_home_station_m,home_access_mode,walk_to_station_min,total_commute_min
0,EMP0001,Poppenbuttel,4271.7,bike,17.1,54.7
1,EMP0002,Harburg,14600.8,drive (park & ride),25.0,90.2
2,EMP0003,Harburg,11804.6,drive (park & ride),20.2,85.4
3,EMP0004,Blankenese,14241.0,drive (park & ride),24.4,78.5
4,EMP0005,Blankenese,4270.0,bike,17.1,71.2
5,EMP0006,Niendorf Nord,1128.6,walk,15.0,50.0
6,EMP0007,Wilhelmsburg,1635.3,bike,6.5,64.0
7,EMP0008,Bergedorf,7295.1,drive (park & ride),12.5,79.9


In [15]:
nearest["total_commute_min"].describe().round(1)

count    300.0
mean      66.8
std       15.0
min       28.3
25%       55.6
50%       67.0
75%       79.5
max       95.5
Name: total_commute_min, dtype: float64

In [16]:
def commute_bucket(minutes):
    if minutes <= 30:
        return "≤ 30 min"
    elif minutes <= 45:
        return "31–45 min"
    elif minutes <= 60:
        return "46–60 min"
    else:
        return "> 60 min"

nearest["commute_bucket"] = nearest["total_commute_min"].apply(commute_bucket)

bucket_counts = nearest["commute_bucket"].value_counts().reindex(
    ["≤ 30 min", "31–45 min", "46–60 min", "> 60 min"]
)
bucket_pct = (bucket_counts / len(nearest) * 100).round(1)

pd.DataFrame({"employees": bucket_counts, "percentage": bucket_pct})

,employees,percentage
commute_bucket,,
≤ 30 min,1,0.3
31–45 min,26,8.7
46–60 min,71,23.7
> 60 min,202,67.3


### Deutschlandticket adoption scoring

To estimate how likely each employee is to adopt the Deutschlandticket, 
this uses a **rule-based weighted score** rather than a machine-learning 
model. A rule-based approach was chosen deliberately: every point in 
the score can be traced back to a specific, explainable factor, which 
is important for a business stakeholder audience "the model 
predicted it" is much harder to justify than "this employee scored 
low because they'd have to drive 8km to the nearest station."

The score (out of 100) is built from three weighted factors:

1. **Commute time (40 points)** — shorter commutes are more attractive 
   for a daily subscription. Weighted highest since time is usually 
   the biggest driver of transport mode choice.
2. **Home access mode (30 points)** — being able to walk or bike to 
   the station is worth more than needing to drive to a park & ride, 
   since driving to catch a train partly cancels out the convenience 
   a subscription is supposed to offer.
3. **Station density near home (30 points)** — having multiple stations 
   within 2km gives more route options and effectively higher service 
   frequency/resilience (e.g. if one line has a delay), which makes 
   transit more dependable and attractive.

These weights are a starting assumption, not a fitted model in a 
real deployment, they could be calibrated against actual survey or 
usage data if it were available.

In [17]:
def score_commute_time(minutes):
    # Shorter commute = more attractive. Score out of 40.
    if minutes <= 30:
        return 40
    elif minutes <= 45:
        return 30
    elif minutes <= 60:
        return 20
    elif minutes <= 90:
        return 10
    else:
        return 0

def score_access_mode(mode):
    # Being able to walk/bike to the station = more attractive than
    # needing to drive (driving to a park & ride partly defeats the
    # convenience appeal of a public transport subscription). Score out of 30.
    return {"walk": 30, "bike": 20, "drive (park & ride)": 5}[mode]

def score_station_density(employee_geom, stations_gdf_m, radius_m=2000):
    # More stations nearby = more route/frequency options = more
    # attractive & resilient transit access. Score out of 30.
    count = (stations_gdf_m.distance(employee_geom) <= radius_m).sum()
    if count >= 3:
        return 30
    elif count == 2:
        return 20
    elif count == 1:
        return 10
    else:
        return 0

nearest["score_commute_time"] = nearest["total_commute_min"].apply(score_commute_time)
nearest["score_access_mode"] = nearest["home_access_mode"].apply(score_access_mode)
nearest["score_station_density"] = nearest.geometry.apply(
    lambda g: score_station_density(g, stations_m)
)

nearest["adoption_score"] = (
    nearest["score_commute_time"]
    + nearest["score_access_mode"]
    + nearest["score_station_density"]
)

nearest[["employee_id", "total_commute_min", "home_access_mode",
         "adoption_score"]].sort_values("adoption_score", ascending=False).head(10)

,employee_id,total_commute_min,home_access_mode,adoption_score
215,EMP0216,42.614635,walk,90
71,EMP0072,37.036368,walk,90
56,EMP0057,53.970818,walk,80
32,EMP0033,59.383970,walk,80
299,EMP0300,46.952214,walk,80
66,EMP0067,44.790219,walk,80
171,EMP0172,51.211855,walk,80
208,EMP0209,59.815618,walk,80
77,EMP0078,53.377899,walk,80
111,EMP0112,41.914644,walk,80


In [18]:
def adoption_tier(score):
    if score >= 70:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

nearest["adoption_potential"] = nearest["adoption_score"].apply(adoption_tier)

tier_summary = nearest["adoption_potential"].value_counts().reindex(["High", "Medium", "Low"])
tier_pct = (tier_summary / len(nearest) * 100).round(1)

pd.DataFrame({"employees": tier_summary, "percentage": tier_pct})

,employees,percentage
adoption_potential,,
High,20,6.7
Medium,65,21.7
Low,215,71.7


In [19]:
# Group by nearest home station to see which areas perform well/poorly
station_summary = nearest.groupby("nearest_home_station").agg(
    employees=("employee_id", "count"),
    avg_commute_min=("total_commute_min", "mean"),
    avg_adoption_score=("adoption_score", "mean"),
).round(1).sort_values("avg_adoption_score", ascending=False)

print("Strongest connectivity areas (top 5):")
display(station_summary.head(5))

print("\nWeakest connectivity areas (bottom 5):")
display(station_summary.tail(5))

Strongest connectivity areas (top 5):


,employees,avg_commute_min,avg_adoption_score
nearest_home_station,,,
Ochsenzoll,1,42.6,90.0
Langenhorn Nord,2,42.9,80.0
Klosterstern,1,51.2,80.0
Ohlsdorf,4,53.4,80.0
Dammtor,2,63.9,75.0



Weakest connectivity areas (bottom 5):


,employees,avg_commute_min,avg_adoption_score
nearest_home_station,,,
Pinneberg,21,66.7,26.7
Rahlstedt,8,63.3,20.6
Blankenese,42,70.3,18.9
Bergedorf,42,81.7,16.8
Harburg,61,83.5,16.1


### Key factors influencing Deutschlandticket adoption

Based on the scoring breakdown, three factors most strongly influence 
adoption likelihood:

1. **Total commute time** — the single biggest driver; commutes over 
   60 minutes make up the majority of the workforce and correlate 
   strongly with Low adoption scores.
2. **Home access mode** — employees who can walk or bike to a station 
   score far higher than those who would need to drive to one, since 
   driving undermines the convenience value of a transit subscription.
3. **Station density near home** — areas with only one nearby station 
   are more vulnerable to delays/disruption reducing reliability, 
   while areas with multiple stations offer more resilient, frequent 
   options.

Overall, this suggests J&J's Norderstedt location itself is the main 
structural constraint: it sits at the edge of the HVV network, so most 
employees living across greater Hamburg face longer combined 
access+transit+last-mile times regardless of promotion efforts. Ticket 
promotion campaigns would likely have the highest ROI if targeted at 
the ~28% of employees in the Medium/High tiers, who live along the 
better-connected U1/S-Bahn corridors.

In [20]:
import folium

# Center map roughly between Hamburg and Norderstedt
m = folium.Map(location=[53.62, 10.0], zoom_start=10, tiles="CartoDB positron")

# Workplace marker
folium.Marker(
    [WORKPLACE["lat"], WORKPLACE["lon"]],
    popup="J&J Medical GmbH (Workplace)",
    icon=folium.Icon(color="red", icon="briefcase", prefix="fa"),
).add_to(m)

# Station markers
for _, row in stations_gdf.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['station_name']} ({row['lines']})",
    ).add_to(m)

# Employee markers, colored by adoption potential
tier_colors = {"High": "green", "Medium": "orange", "Low": "gray"}
for _, row in nearest.iterrows():
    folium.CircleMarker(
        [row["home_lat"], row["home_lon"]],
        radius=3,
        color=tier_colors[row["adoption_potential"]],
        fill=True,
        fill_opacity=0.6,
        popup=f"{row['employee_id']} — {row['adoption_potential']} ({row['adoption_score']} pts, {row['total_commute_min']:.0f} min)",
    ).add_to(m)

m.save("commute_map.html")
m